<a href="https://colab.research.google.com/github/Pgiuliano8/COMS5790-NLP-Projects/blob/main/project4/Project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q --force-reinstall --no-cache-dir "transformers==4.48.3" "sentence-transformers==3.4.1" "sentencepiece" "sacrebleu" "rouge-score"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.5 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
datasets 4.8.5 requires fs

In [2]:
import csv
import itertools
import json
import random
from pathlib import Path

ANIMAL_PATH = Path('/content/animal.tsv')
LITCOVID_PATH = Path('/content/litcovid_sample_1000.json')
SEED = 42


def clean_text(value):
    return ' '.join(value.split()) if isinstance(value, str) else ''


def make_record(title, abstract):
    title, abstract = clean_text(title), clean_text(abstract)
    return {'abstract': abstract, 'title': title} if title and abstract else None


def load_animal(path):
    if not path.is_file():
        raise FileNotFoundError(f'Animal TSV not found: {path}')
    with path.open(encoding='utf-8-sig', newline='') as file:
        reader = csv.reader(file, delimiter='\t')
        first_row = next(reader, None)
        if first_row is None:
            raise ValueError(f'Animal TSV is empty: {path}')

        header = [item.strip().lstrip('\ufeff').casefold() for item in first_row]
        if 'title' in header and 'abstract' in header:
            title_i, abstract_i = header.index('title'), header.index('abstract')
            rows = reader
        else:
            # Headerless Animal rows are normally ID, Title, Abstract.
            title_i, abstract_i = (1, 2) if len(first_row) != 2 else (0, 1)
            rows = itertools.chain([first_row], reader)

        records = []
        for row in rows:
            if max(title_i, abstract_i) >= len(row):
                continue
            record = make_record(row[title_i], row[abstract_i])
            if record:
                records.append(record)
    if not records:
        raise ValueError('No Animal records with both title and abstract were found.')
    return records


def load_litcovid(path):
    if not path.is_file():
        raise FileNotFoundError(f'LitCovid JSON not found: {path}')
    with path.open(encoding='utf-8-sig') as file:
        payload = json.load(file)

    if isinstance(payload, list):
        documents = payload
    elif isinstance(payload, dict) and isinstance(payload.get('documents'), list):
        documents = payload['documents']
    elif isinstance(payload, dict) and isinstance(payload.get('passages'), list):
        documents = [payload]
    else:
        raise ValueError('Unrecognized BioC JSON format.')

    records = []
    for document in documents:
        if not isinstance(document, dict):
            continue
        titles, abstracts = [], []
        for passage in document.get('passages', []):
            if not isinstance(passage, dict):
                continue
            infons = passage.get('infons', {})
            if not isinstance(infons, dict):
                continue
            section = ''
            for key in ('section_type', 'section', 'type'):
                value = str(infons.get(key, '')).strip().upper()
                if value in ('TITLE', 'ABSTRACT'):
                    section = value
                    break
            text = clean_text(passage.get('text'))
            if section == 'TITLE' and text:
                titles.append(text)
            elif section == 'ABSTRACT' and text:
                abstracts.append(text)
        record = make_record(' '.join(titles), ' '.join(abstracts))
        if record:
            records.append(record)
    if not records:
        raise ValueError('No LitCovid records with both title and abstract were found.')
    return records


def split_records(records, seed=SEED):
    if len(records) < 3:
        raise ValueError(f'Need at least 3 records to split; found {len(records)}.')
    ratios = {'train': 0.4, 'validation': 0.2, 'test': 0.4}
    exact = {name: len(records) * ratio for name, ratio in ratios.items()}
    sizes = {name: int(size) for name, size in exact.items()}
    remainder = len(records) - sum(sizes.values())
    order = sorted(ratios, key=lambda name: (-(exact[name] - sizes[name]), list(ratios).index(name)))
    for name in order[:remainder]:
        sizes[name] += 1
    shuffled = list(records)
    random.Random(seed).shuffle(shuffled)
    train_end = sizes['train']
    val_end = train_end + sizes['validation']
    return {
        'train': shuffled[:train_end],
        'validation': shuffled[train_end:val_end],
        'test': shuffled[val_end:],
    }


animal_records = load_animal(ANIMAL_PATH)
litcovid_records = load_litcovid(LITCOVID_PATH)

animal_splits = split_records(animal_records)
litcovid_splits = split_records(litcovid_records)

for name, records, splits in (
    ('Animal', animal_records, animal_splits),
    ('LitCovid', litcovid_records, litcovid_splits),
):
    print(name, '| usable records:', len(records), '| splits:',
          {key: len(value) for key, value in splits.items()})
    print('Example:', records[0]['title'][:100])

Animal | usable records: 1108 | splits: {'train': 443, 'validation': 222, 'test': 443}
Example: Whole-genome association and genome partitioning revealed variants and explained heritability for to
LitCovid | usable records: 730 | splits: {'train': 292, 'validation': 146, 'test': 292}
Example: Side effects of ruxolitinib in patients with SARS-CoV-2 infection: Two case reports References


In [3]:
import gc
import math
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    set_seed,
)

MODEL_NAME = 'google/flan-t5-small'
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 64
BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 2e-5
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
trained_models = {}


def encode_records(records, tokenizer):
    encoded = []
    for record in records:
        source = tokenizer(
            'Generate a concise scientific title: ' + record['abstract'],
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
        )
        target = tokenizer(
            text_target=record['title'],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
        )['input_ids']
        if not source['input_ids'] or not target:
            raise ValueError('Found an empty tokenized abstract or title.')
        encoded.append({
            'input_ids': source['input_ids'],
            'attention_mask': source['attention_mask'],
            'labels': target,
        })
    return encoded


def move_batch(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def get_validation_loss(model, loader, name):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch)
            n_tokens = int((batch['labels'] != -100).sum().item())
            if n_tokens == 0:
                raise RuntimeError(f'{name}: validation batch has no target tokens.')
            loss = model(**batch).loss
            if not torch.isfinite(loss).item():
                raise RuntimeError(f'{name}: validation loss became NaN/Inf.')
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens
    return total_loss / total_tokens


def train_one_dataset(name, splits):
    for split in ('train', 'validation', 'test'):
        if split not in splits or not splits[split]:
            raise ValueError(f'{name}: {split} split is empty.')

    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100,
    )

    train_data = encode_records(splits['train'], tokenizer)
    validation_data = encode_records(splits['validation'], tokenizer)
    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        generator=torch.Generator().manual_seed(SEED),
    )
    validation_loader = DataLoader(
        validation_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
    )

    # Check the pretrained model's loss before changing any weights.
    model.eval()
    first_batch = move_batch(collator(train_data[:2]))
    with torch.no_grad():
        initial_loss = model(**first_batch).loss.item()
    print(f'{name} initial loss: {initial_loss:.4f}')
    if not math.isfinite(initial_loss):
        raise RuntimeError(f'{name}: initial loss is NaN/Inf; training was not started.')

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum, train_tokens = 0.0, 0

        for step, batch in enumerate(train_loader, start=1):
            batch = move_batch(batch)
            n_tokens = int((batch['labels'] != -100).sum().item())
            if n_tokens == 0:
                raise RuntimeError(f'{name}: training batch has no target tokens.')

            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            if not torch.isfinite(loss).item():
                raise RuntimeError(f'{name}: training loss became NaN/Inf at batch {step}.')

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm).item():
                raise RuntimeError(f'{name}: gradients became NaN/Inf at batch {step}.')
            optimizer.step()

            train_loss_sum += loss.item() * n_tokens
            train_tokens += n_tokens

        train_loss = train_loss_sum / train_tokens
        val_loss = get_validation_loss(model, validation_loader, name)
        print(
            f'{name} | epoch {epoch}/{EPOCHS} | '
            f'train loss {train_loss:.4f} | validation loss {val_loss:.4f}'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

    if best_state is None or not math.isfinite(best_val_loss):
        raise RuntimeError(f'{name}: no valid validation checkpoint was found.')

    model.load_state_dict(best_state)
    model.to('cpu')
    trained_models[name] = (model, tokenizer)
    print(f'{name}: best validation loss {best_val_loss:.4f}; model kept in memory.')

    del optimizer, train_loader, validation_loader, train_data, validation_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


train_one_dataset('animal', animal_splits)
train_one_dataset('litcovid', litcovid_splits)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


animal initial loss: 2.6709
animal | epoch 1/3 | train loss 2.3099 | validation loss 1.9223
animal | epoch 2/3 | train loss 2.1651 | validation loss 1.8761
animal | epoch 3/3 | train loss 2.0736 | validation loss 1.8395
animal: best validation loss 1.8395; model kept in memory.
litcovid initial loss: 2.6265
litcovid | epoch 1/3 | train loss 2.8413 | validation loss 2.3407
litcovid | epoch 2/3 | train loss 2.6876 | validation loss 2.2691
litcovid | epoch 3/3 | train loss 2.5766 | validation loss 2.2243
litcovid: best validation loss 2.2243; model kept in memory.


In [4]:
import math
import re
from collections import Counter

import torch
from sentence_transformers import SentenceTransformer, util

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 8
MAX_NEW_TOKENS = 64
SEMANTIC_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'


def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.casefold(), flags=re.UNICODE)


def count_ngrams(tokens, order):
    return Counter(tuple(tokens[i:i + order]) for i in range(len(tokens) - order + 1))


def corpus_bleu(predictions, references):
    matches, totals = [0] * 4, [0] * 4
    pred_length = ref_length = 0
    for prediction, reference in zip(predictions, references):
        pred, ref = tokenize(prediction), tokenize(reference)
        pred_length += len(pred)
        ref_length += len(ref)
        for order in range(1, 5):
            pred_ngrams = count_ngrams(pred, order)
            ref_ngrams = count_ngrams(ref, order)
            totals[order - 1] += sum(pred_ngrams.values())
            matches[order - 1] += sum(
                min(count, ref_ngrams[gram]) for gram, count in pred_ngrams.items()
            )
    if pred_length == 0:
        return 0.0
    precisions, smooth = [], 1.0
    for match, total in zip(matches, totals):
        if total == 0:
            continue
        if match:
            precisions.append(match / total)
        else:
            smooth *= 2
            precisions.append(1 / (smooth * total))
    if not precisions:
        return 0.0
    brevity = 1.0 if pred_length > ref_length else math.exp(1 - ref_length / pred_length)
    return 100 * brevity * math.exp(sum(math.log(p) for p in precisions) / len(precisions))


def f1(overlap, pred_count, ref_count):
    if overlap == 0 or pred_count == 0 or ref_count == 0:
        return 0.0
    precision, recall = overlap / pred_count, overlap / ref_count
    return 2 * precision * recall / (precision + recall)


def rouge2_f1(prediction, reference):
    pred = count_ngrams(tokenize(prediction), 2)
    ref = count_ngrams(tokenize(reference), 2)
    overlap = sum(min(count, ref[gram]) for gram, count in pred.items())
    return f1(overlap, sum(pred.values()), sum(ref.values()))


def lcs_length(first, second):
    previous = [0] * (len(second) + 1)
    for token in first:
        current = [0]
        for index, other in enumerate(second, start=1):
            current.append(
                previous[index - 1] + 1
                if token == other
                else max(previous[index], current[-1])
            )
        previous = current
    return previous[-1]


def rougeL_f1(prediction, reference):
    pred, ref = tokenize(prediction), tokenize(reference)
    return f1(lcs_length(pred, ref), len(pred), len(ref))


def generate_test_titles(model, tokenizer, records):
    model.to(DEVICE)
    model.eval()
    generated_titles = []
    with torch.inference_mode():
        for start in range(0, len(records), BATCH_SIZE):
            batch_records = records[start:start + BATCH_SIZE]
            inputs = tokenizer(
                [
                    'Generate a concise scientific title: ' + item['abstract']
                    for item in batch_records
                ],
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512,
            ).to(DEVICE)
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=4,
            )
            generated_titles.extend(
                tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            )
    model.to('cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return [title.strip() for title in generated_titles]


semantic_model = SentenceTransformer(SEMANTIC_MODEL_NAME, device='cpu')

for dataset_name, splits in (
    ('animal', animal_splits),
    ('litcovid', litcovid_splits),
):
    model, tokenizer = trained_models[dataset_name]
    test_records = splits['test']
    references = [record['title'] for record in test_records]
    predictions = generate_test_titles(model, tokenizer, test_records)
    if not references or len(predictions) != len(references):
        raise RuntimeError(f'{dataset_name}: generated predictions do not match test records.')

    pred_embeddings = semantic_model.encode(
        predictions, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    ref_embeddings = semantic_model.encode(
        references, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    semantic_similarity = util.cos_sim(pred_embeddings, ref_embeddings).diagonal().mean().item()

    scores = {
        'BLEU (0-100)': corpus_bleu(predictions, references),
        'ROUGE-2 F1 (0-1)': sum(
            rouge2_f1(pred, ref) for pred, ref in zip(predictions, references)
        ) / len(references),
        'ROUGE-L F1 (0-1)': sum(
            rougeL_f1(pred, ref) for pred, ref in zip(predictions, references)
        ) / len(references),
        'Semantic cosine similarity (-1 to 1)': float(semantic_similarity),
    }
    print(f'\n{dataset_name.upper()} — test examples: {len(test_records)}')
    for metric, value in scores.items():
        print(f'{metric}: {value:.4f}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)



ANIMAL — test examples: 443
BLEU (0-100): 18.2291
ROUGE-2 F1 (0-1): 0.2295
ROUGE-L F1 (0-1): 0.4026
Semantic cosine similarity (-1 to 1): 0.7161

LITCOVID — test examples: 292
BLEU (0-100): 18.2294
ROUGE-2 F1 (0-1): 0.2245
ROUGE-L F1 (0-1): 0.3495
Semantic cosine similarity (-1 to 1): 0.6882


In [5]:
import gc
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


def evaluate_predictions(predictions, references):
    pred_emb = semantic_model.encode(
        predictions, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    ref_emb = semantic_model.encode(
        references, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    sim_scores = util.cos_sim(pred_emb, ref_emb).diagonal().tolist()
    rouge2_scores = [rouge2_f1(p, r) for p, r in zip(predictions, references)]
    rougeL_scores = [rougeL_f1(p, r) for p, r in zip(predictions, references)]

    summary = {
        'BLEU (0-100)': corpus_bleu(predictions, references),
        'ROUGE-2 F1': sum(rouge2_scores) / len(references),
        'ROUGE-L F1': sum(rougeL_scores) / len(references),
        'Semantic Sim': sum(sim_scores) / len(references),
    }
    return summary, rougeL_scores, sim_scores


# Load the base pretrained model once for zero-shot evaluation
zs_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
zs_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

for dataset_name, splits in (
    ('animal', animal_splits),
    ('litcovid', litcovid_splits),
):
    test_records = splits['test']
    references = [record['title'] for record in test_records]

    # 1. Zero-shot evaluation
    zs_preds = generate_test_titles(zs_model, zs_tokenizer, test_records)
    zs_summary, _, _ = evaluate_predictions(zs_preds, references)

    # 2. Fine-tuned evaluation
    ft_model, ft_tokenizer = trained_models[dataset_name]
    ft_preds = generate_test_titles(ft_model, ft_tokenizer, test_records)
    ft_summary, ft_rougeL, ft_sim = evaluate_predictions(ft_preds, references)

    print(f"\n{'='*60}")
    print(f"{dataset_name.upper()} — Zero-Shot vs. Fine-Tuned (n={len(test_records)})")
    print(f"{'='*60}")
    print(f"{'Metric':<18} | {'Zero-Shot':>10} | {'Fine-Tuned':>10} | {'Delta':>10}")
    print("-" * 58)
    for metric in zs_summary:
        zs_val = zs_summary[metric]
        ft_val = ft_summary[metric]
        print(f"{metric:<18} | {zs_val:>10.4f} | {ft_val:>10.4f} | {ft_val - zs_val:>+10.4f}")

    # 3. Select qualitative examples for error and semantic-gap analysis
    indices = list(range(len(test_records)))
    best_idx = max(indices, key=lambda i: ft_sim[i])
    worst_idx = min(indices, key=lambda i: ft_sim[i])
    gap_idx = max(indices, key=lambda i: ft_sim[i] - ft_rougeL[i])

    print("\n--- Qualitative Examples ---")
    for label, idx in (
        ("HIGH SEMANTIC MATCH", best_idx),
        ("LOW SEMANTIC MATCH", worst_idx),
        ("HIGH SEMANTIC / LOW ROUGE-L (Paraphrase)", gap_idx),
    ):
        print(f"\n[{label}] (ROUGE-L: {ft_rougeL[idx]:.3f} | Semantic Sim: {ft_sim[idx]:.3f})")
        print(f"  Ground Truth : {references[idx]}")
        print(f"  Fine-Tuned   : {ft_preds[idx]}")
        print(f"  Zero-Shot    : {zs_preds[idx]}")

del zs_model, zs_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


ANIMAL — Zero-Shot vs. Fine-Tuned (n=443)
Metric             |  Zero-Shot | Fine-Tuned |      Delta
----------------------------------------------------------
BLEU (0-100)       |    10.6300 |    18.2291 |    +7.5991
ROUGE-2 F1         |     0.1553 |     0.2295 |    +0.0742
ROUGE-L F1         |     0.2770 |     0.4026 |    +0.1256
Semantic Sim       |     0.6513 |     0.7161 |    +0.0647

--- Qualitative Examples ---

[HIGH SEMANTIC MATCH] (ROUGE-L: 0.884 | Semantic Sim: 0.998)
  Ground Truth : Polymorphisms in the ovine myostatin gene are associated with birth weight but not with weight gain in Iranian Makoei sheep.
  Fine-Tuned   : Polymorphism in the ovine myostatin gene is associated with birth weight, but not with weight gain in Iranian Makoei sheep.
  Zero-Shot    : Polymorphism in the ovine myostatin gene is associated with birth weight, but not with weight gain in Iranian Makoei sheep

[LOW SEMANTIC MATCH] (ROUGE-L: 0.071 | Semantic Sim: 0.079)
  Ground Truth : Association ana